# Setup

## Imports and Installs

### Installs
No need to install OSMPythonTools as it is no longer used.

In [ ]:
!pip install geopandas
!pip install geojson
!pip install overpass
!pip install alphashape

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.6/507.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 735.5/735.5 kB 35.2 MB/s eta 0:00:00


### Imports

In [ ]:
import pandas as pd
import json
import geojson
#https://github.com/mvexel/overpass-api-python-wrapper/blob/main/README.md
import overpass
#https://pypi.org/project/osm2geojson/
import csv
import os
import logging
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from shapely.geometry import Point, Polygon
import requests
import time
from google.colab import drive
import sys
import ast
from alphashape import alphashape
import geopandas as gpd
import os
from collections import defaultdict
import os.path
from tqdm import tqdm

## Parameters

In [ ]:
## super spot hotspot ratio is the average number of census tracts per cluster, to get number of expected super spots do census tracts # / superspot ratio
## min children is just the minimum number of census tracts needed for a cluster otherwsie the cluster will be removed and reallocated


superspot_hotspot_ratio = 10
min_children = superspot_hotspot_ratio - 3


In [ ]:
#State name to be taken for input later
#state_name = input()
state_name = "Ohio"


#City name to be taken for later input
#city_name = input()
city_name = "Columbus"

In [ ]:
re_gen = True

## Image Utilities Setup

In [ ]:
# Default size of images
size = (16,16)

# Function to check for img directory and save image
def save_image(plot, img_name: str):
  try:
    os.mkdir(data_dir + "/Images")
  except:
    print(data_dir + "/Images" + " Already Exists")

  try:
    os.mkdir(f'{data_dir}/Images/Data-{min_children}-{superspot_hotspot_ratio}')
  except:
    print(f'{data_dir}/Images/Data-{min_children}-{superspot_hotspot_ratio}', "Already Exists")

  plot.savefig(f'{data_dir}/Images/Data-{min_children}-{superspot_hotspot_ratio}/{img_name}.png')

## Connecting to Drive

We first mount the drive. This step requires permission to be given. Running the cell should open a permission page to allow colab access to google drive.

In [ ]:
drive.mount('/content/gdrive/', force_remount=True)

Mounted at /content/gdrive/


Setting the file directory to the data folder. All files will be referenced to this working directory.

Folder structure:

```
📁Drive/
└─ 📁<personal_dir>/
   ├─ 📁<city_name_1> - RL Delivery Data/
   ├─ 📁<city_name_2> - RL Delivery Data/
   └─ 📁Raw GeoJsons/
```


In [ ]:
if state_name == "District of Columbia":
  city_name = "District of Columbia"

# Your personal folder to STORE THE FOLDER THAT WILL STORE DATA goes here
personal_dir = "./gdrive/MyDrive/DeliverAI Data Folder/"

# personal_dir = "./gdrive/MyDrive/Thesis/Source Code & Supporting Files/City Data"  # Example personal folder for Robert, assuming "City Data" folder is created in drive

mini = True
data_dir = f"{personal_dir}{city_name}_mini - RL Delivery Data" if mini else f"{personal_dir}{city_name} - RL Delivery Data"

try:
  os.mkdir(data_dir)
except:
  print(data_dir + " Already Exists")

# See directory content
data = os.listdir(data_dir)
print("Files in directory : ", data)


./gdrive/MyDrive/DeliverAI Data Folder/Columbus_mini - RL Delivery Data Already Exists
Files in directory :  ['Census Data', 'Original Location Data', 'Processed Location Data', 'Images', 'Hotspot Data', 'Q Tables', 'avg_hotspot_data.json', 'ExecutionTime_7_31.txt', 'UMST Graph', 'Deliveries', 'Graphing']


In [ ]:
# Chicago (31 census tracts)
census_tract_subset = ['17031062900', '17031071400', '17031062200', '17031060900', '17031063303', '17031061800', '17031070102', '17031071600', '17031063301', '17031071300', '17031071100', '17031832600', '17031832000', '17031063100', '17031071500', '17031061901', '17031063000', '17031070103', '17031062000', '17031063200', '17031063400', '17031070200', '17031070400', '17031071800', '17031063302', '17031071200', '17031070300', '17031061902', '17031070101', '17031071000', '17031062100']

# Columbus
# census_tract_subset = ['39049001300', '39049001400', '39049002000', '39049002510', '39049002520', '39049007830', '39049002100', '39049001000', '39049001810', '39049001902', '39049002800', '39049003200', '39049003600', '39049008500', '39049001110', '39049001200', '39049001500', '39049001700', '39049001820', '39049002200', '39049002300', '39049001901' '39049001122', '39049003000', '39049002900', '39049001600', '39049001121', '39049001901']

# Philadelphia
# census_tract_subset = ['42101008802', '42101008702', '42101008701', '42101008602', '42101008601', '42101008500', '42101008000', '42101007900', '42101009400', '42101009300', '42101009200', '42101010400', '42101010300', '42101010500', '42101010200', '42101008400', '42101007200', '42101008102', '42101008101', '42101007800', '42101007700', '42101007300', '42101011100', '42101011200', '42101019500', '42101011800', '42101011300', '42101010700', '42101010600', '42101011900']

# Getting Consumer and Producer data


Using OSM, we can query locations that are suited for our study. We need those locations which can either provide food or act as a potential drop location for a customer. Thse locations can be queried using Overpass API that uses the OSM data. While it is possible to use the OSMPythonTools for queries the data uses python scripts, it is easier to query and visualise data from Overpass Turbo and directly download the data from the site in the `.geojson` format. The latter was used for this notebook but now they are directly gotten using Overpass API. These data files are stored in `Original Data` folder.

### Getting the Pickup and Drop points

In [ ]:
#Overpass API example Query - NOTE approaches used through overpass turbo may not work directly through overpass API due to overpass turbo
#allowing additional query struture for ease of use. This can be overcome by taking a query crafted using overpass Turbo and going to
#Export > Query > Standalone Query > download/copy to get the actual query that overpass turbo sends to the overpass API

"""  OLD VERSION OF QUERY
[out:json][timeout:900];
// fetch area “chicago” to search in
area[name="Chicago"]->.searchArea;
(
  way["building"="dormitory"](area.searchArea);
  //relation["building"="dormitory"](area.searchArea);

);
out body;
>;
out skel qt;
"""

# Admin levels: 2: country, 4: state, 8: cities
"""  NEW VERSION OF QUERY
[out:json][timeout:900];

area[name="United States"]["admin_level"="2"]->.country;
area[name="Ohio"]["admin_level"="4"]->.state;
area[name="Oxford"]->.city;

way["building"="dormitory"](area.country)(area.state)(area.city);

out body;
>;
out skel qt;
"""

## Implement this in the code below?
"""  NEW VERSION OF QUERY
[out:json][timeout:900];

area[name="United States"]["admin_level"="2"]->.country;
area[name="Ohio"]["admin_level"="4"]->.state;
area[name="Oxford"]->.city;
(
  way["building"="dormitory"](area.country)(area.state)(area.city);
  node[\"{category}\"=\"{tag}\"](area.country)(area.state)(area.city);
);

out body;
>;
out skel qt;
"""

'  NEW VERSION OF QUERY\n[out:json][timeout:900];\n\narea[name="United States"]["admin_level"="2"]->.country;\narea[name="Ohio"]["admin_level"="4"]->.state;\narea[name="Oxford"]->.city;\n(\n  way["building"="dormitory"](area.country)(area.state)(area.city);\n  node["{category}"="{tag}"](area.country)(area.state)(area.city);\n);\n\nout body;\n>;\nout skel qt;\n'

Creates a query from given attributes

In [ ]:
# Timeout increased from 25 to 900

## OLD
# def buildQuery(stateName, cityName, category, tag):
#   return f"""
          # [out:json][timeout:900];
          # area[name=\"United States\"][\"admin_level\"=\"2\"]->.country;
          # area[name=\"{stateName}\"][\"admin_level\"=\"4\"]->.state;
          # area[name=\"{cityName}\"]->.city;
          # (
          #   way[\"{category}\"=\"{tag}\"](area.country)(area.state)(area.city);
          # );
          # out body;
          # >;
          # out skel qt;
#           """

def buildQuery(stateName, cityName, category, tag, is_dc=False):
  if is_dc:
    return f"""
          [out:json][timeout:3600];
          area[name=\"United States\"][\"admin_level\"=\"2\"]->.country;
          area[name=\"{stateName}\"][\"admin_level\"=\"4\"]->.state;
          (
            way[\"{category}\"=\"{tag}\"](area.country)(area.state);
            node[\"{category}\"=\"{tag}\"](area.country)(area.state);
          );
          out body;
          >;
          out skel qt;
          """
  else:
    return f"""
          [out:json][timeout:3600];
          area[name=\"United States\"][\"admin_level\"=\"2\"]->.country;
          area[name=\"{stateName}\"][\"admin_level\"=\"4\"]->.state;
          area[name=\"{cityName}\"]->.city;
          (
            way[\"{category}\"=\"{tag}\"](area.country)(area.state)(area.city);
            node[\"{category}\"=\"{tag}\"](area.country)(area.state)(area.city);
          );
          out body;
          >;
          out skel qt;
          """

# Test print to make sure it works
#print(buildQuery(state_name, city_name, "building", "dormitory"))

In [ ]:
#Converts the raw osm data into a geojson object, this object needs to be in the same form that the original data gathered @
# https://overpass-turbo.eu/ is in. More information about the geojson can be found in the help section of overpass turbo.

## OLD
# def overpass_to_geojson(overpass_data):
#   node_dict = {node['id']: node for node in overpass_data['elements'] if node['type'] == 'node'}
#   features = []
#   for element in overpass_data['elements']:
#     if element['type'] == 'way' and 'tags' in element and ('building' in element['tags'] or 'building:part' in element['tags']):
#       coordinates = [[node_dict[node_id]['lon'], node_dict[node_id]['lat']] for node_id in element['nodes']]
#       # Ensure the polygon is closed by repeating the first coordinate at the end
#       if coordinates[0] != coordinates[-1]:
#         coordinates.append(coordinates[0])
#       geometry = {'type': 'Polygon', 'coordinates': [coordinates]}
#       properties = {'@id': f"way/{element['id']}", **element['tags']}  # Include tags directly within properties
#       feature = {'type': 'Feature', 'id': f"way/{element['id']}", 'properties': properties, 'geometry': geometry}  # Include 'id' here
#       features.append(feature)
#   feature_collection = {
#     'type': 'FeatureCollection',
#     'features': features,
#     'generator': 'overpass-turbo',  # Added generator field
#     'copyright': 'The data included in this document is from www.openstreetmap.org. The data is made available under ODbL.',  # Added copyright field
#     'timestamp': '2024-06-02T00:20:11Z'  # Added timestamp field
#   }
#   return feature_collection

def overpass_to_geojson(overpass_data):
  node_dict = {node['id']: node for node in overpass_data['elements'] if node['type'] == 'node'}
  features = []
  for element in overpass_data['elements']:
    if 'tags' in element.keys():
      if element['type'] == 'way':
        coordinates = [[node_dict[node_id]['lon'], node_dict[node_id]['lat']] for node_id in element['nodes']]
        # Ensure the polygon is closed by repeating the first coordinate at the end
        if coordinates[0] != coordinates[-1]:
          coordinates.append(coordinates[0])
        geometry = {'type': 'Polygon', 'coordinates': [coordinates]}
        properties = {'@id': f"way/{element['id']}", **element['tags']}  # Include tags directly within properties
        feature = {'type': 'Feature', 'id': f"way/{element['id']}", 'properties': properties, 'geometry': geometry}  # Include 'id' here
        features.append(feature)
      if element['type'] == 'node':
        coordinates = [[element['lon'], element['lat']]]
        geometry = {'type': 'Point', 'coordinates': [coordinates]}
        properties = {'@id': f"node/{element['id']}", **element['tags']}
        feature = {'type': 'Feature', 'id': f"node/{element['id']}", 'properties': properties, 'geometry': geometry}
        features.append(feature)
  feature_collection = {
    'type': 'FeatureCollection',
    'features': features,
    'generator': 'overpass-turbo',  # Added generator field
    'copyright': 'The data included in this document is from www.openstreetmap.org. The data is made available under ODbL.',  # Added copyright field
    'timestamp': '2024-06-02T00:20:11Z'  # Added timestamp field
  }
  return feature_collection


In [ ]:
# sample_query = buildQuery(state_name, city_name, "building", "detached")

# testing_overpass_api = overpass.API(timeout=3600)
# result = testing_overpass_api.get(sample_query, build=False)
# sample_ovp_data = overpass_to_geojson(result)


# len([e for e in result['elements'] if 'tags' in e.keys()])
# # len(sample_ovp_data['features'])


In [ ]:
def save_data(query, fileName):
  query_sub = query[query.index("("):]
  print("Query: " + query_sub[7:query_sub.index("]")+1])

  # Create an Overpass API object
  api = overpass.API(timeout=3600)

  #This took hours to find out, if build is not set to false by default it tries to add its own "[out:json];" and "out body;" at the begining and end
  result = api.get(query, build=False)
  print("Result done.")


  if(fileName in pickLocations):
    modifier = "Original Location Data/Pickup Point Data"
  else:
    modifier = "Original Location Data/Drop Point Data"


  data = overpass_to_geojson(result)
  print("Count:", len(data['features']))


  with open(data_dir + "/" + modifier + "/" + fileName, 'w') as f:
    json.dump(data, f, indent=2)
  print("Dumped.", end="\n------\n")


In [ ]:
fileNames = ["restaurant_raw_data.geojson","cafe_raw_data.geojson","apartments_raw_data.geojson","residential_raw_data.geojson","fast_food_raw_data.geojson","food_court_raw_data.geojson","college_raw_data.geojson","university_raw_data.geojson","dormitory_raw_data.geojson","hospital_raw_data.geojson","detached_raw_data.geojson","hotel_raw_data.geojson","house_raw_data.geojson","office_raw_data.geojson","community_centre_raw_data.geojson","bakery_raw_data.geojson","supermarket_raw_data.geojson"]
pickLocations = ["supermarket_raw_data.geojson","restaurant_raw_data.geojson","cafe_raw_data.geojson","fast_food_raw_data.geojson","food_court_raw_data.geojson","bakery_raw_data.geojson"]
dropLocations = ["apartments_raw_data.geojson","residential_raw_data.geojson","college_raw_data.geojson","university_raw_data.geojson","dormitory_raw_data.geojson","hospital_raw_data.geojson","detached_raw_data.geojson","hotel_raw_data.geojson","house_raw_data.geojson","office_raw_data.geojson","community_centre_raw_data.geojson"]


try:
  os.mkdir(data_dir + "/" + "Original Location Data")
except:
  print(data_dir + "/" + "Original Location Data" + " Already Exists")

try:
  os.mkdir(data_dir + "/" + "Original Location Data/Pickup Point Data")
except:
  print(data_dir + "/" + "Original Location Data/Pickup Point Data" + " Already Exists")

try:
  os.mkdir(data_dir + "/" + "Original Location Data/Drop Point Data")
except:
  print(data_dir + "/" + "Original Location Data/Drop Point Data" + " Already Exists")

try:
  os.mkdir(data_dir + "/" + "Processed Location Data")
except:
  print(data_dir + "/" + "Processed Location Data" + " Already Exists")

try:
  os.mkdir(data_dir + "/" + "Processed Location Data/Pickup Point Data")
except:
  print(data_dir + "/" + "Processed Location Data/Pickup Point Data" + " Already Exists")

try:
  os.mkdir(data_dir + "/" + "Processed Location Data/Drop Point Data")
except:
  print(data_dir + "/" + "Processed Location Data/Drop Point Data" + " Already Exists")


consumer_producer_data_exists = True

for fName in pickLocations:
  if (not os.path.isfile(data_dir + "/Original Location Data/Pickup Point Data/" + fName)):
    consumer_producer_data_exists = False


for fName in pickLocations:
  if (not os.path.isfile(data_dir + "/Original Location Data/Drop Point Data/" + fName)):
    consumer_producer_data_exists = False


if(consumer_producer_data_exists == False or re_gen):


  queries = [
    buildQuery(state_name, city_name, "amenity", "restaurant"),
    buildQuery(state_name, city_name, "amenity", "cafe"),
    buildQuery(state_name, city_name, "building", "apartments"),
    buildQuery(state_name, city_name, "building", "residential"),
    buildQuery(state_name, city_name, "amenity", "fast_food"),
    buildQuery(state_name, city_name, "amenity", "food_court"),
    buildQuery(state_name, city_name, "building", "college"),
    buildQuery(state_name, city_name, "building", "university"),
    buildQuery(state_name, city_name, "building", "dormitory"),
    buildQuery(state_name, city_name, "building", "hospital"),
    buildQuery(state_name, city_name, "building", "detached"),
    buildQuery(state_name, city_name, "building", "hotel"),
    buildQuery(state_name, city_name, "building", "house"),
    buildQuery(state_name, city_name, "building", "office"),
    buildQuery(state_name, city_name, "amenity", "community_centre"),
    buildQuery(state_name, city_name, "shop", "bakery"),
    buildQuery(state_name, city_name, "shop", "supermarket")
  ]

  for i, query in enumerate(queries):
      try:
          save_data(query, fileNames[i])
          time.sleep(10)  # wait 10 seconds before next request
      except Exception as e:
          print(f"Error on query {i}: {e}")
          time.sleep(30)  # longer wait if it fails
else:
  print("Consumer/Producer Data already exists.")  # TODO CHANGE LATER

./gdrive/MyDrive/DeliverAI Data Folder/Columbus_mini - RL Delivery Data/Original Location Data Already Exists
./gdrive/MyDrive/DeliverAI Data Folder/Columbus_mini - RL Delivery Data/Original Location Data/Pickup Point Data Already Exists
./gdrive/MyDrive/DeliverAI Data Folder/Columbus_mini - RL Delivery Data/Original Location Data/Drop Point Data Already Exists
./gdrive/MyDrive/DeliverAI Data Folder/Columbus_mini - RL Delivery Data/Processed Location Data Already Exists
./gdrive/MyDrive/DeliverAI Data Folder/Columbus_mini - RL Delivery Data/Processed Location Data/Pickup Point Data Already Exists
./gdrive/MyDrive/DeliverAI Data Folder/Columbus_mini - RL Delivery Data/Processed Location Data/Drop Point Data Already Exists
Query:        way["amenity"="restaurant"]
Result done.
Count: 778
Dumped.
------
Query:        way["amenity"="cafe"]
Result done.
Count: 194
Dumped.
------
Query:        way["building"="apartments"]
Result done.
Count: 9611
Dumped.
------
Query:        way["building"="

#Creating the GeoJSON


Geojson provides easy and effecient to handle geo data. It has only one main requirement - every data frame element needs to have an attribute that contains geometry objects. Usually it is named as `geometry` or needs to be specified seperately if the name is different.

### Drop Unneccsary Attributes
First drop unnecessary attributes. The Census Tract data contains many attributes that are not necessary for our analysis. We drop these columns to avoid storing excess data and taking extra time for computing queries. This becomes specially important when we have to perform joins among data frames.

In [ ]:
#Edited to allow for generic city use

"""
INSERT CHECK FOR data_dir + "/Census Data/RAW_census_tract_data.geojson"
"""


# census_dir = data_dir + "/Census Data/RAW GeoJsons/" + state_name + ".geojson"

# IMPORTANT: I store the GeoJsons in one folder outside of the data directory of individual cities so that I only need to have it once. The original line of code is above.
census_dir = personal_dir + "Raw GeoJsons/" + state_name + ".geojson"
census_df = gpd.read_file(census_dir)

#All attributes of a geojson converted from census.gov shapefiles using ogr2ogr
#keeping GEOID as it represents a unique identifier for each census tract. It is a combination of the census ID which is only unqiue within
#a county, county ID which is only unique within a state, and the state ID. This makes a fully unique idetifier that is 11 digits long.
census_df = census_df.drop('STATEFP', axis=1)
census_df = census_df.drop('COUNTYFP', axis=1)
census_df = census_df.drop('TRACTCE', axis=1)
census_df = census_df.drop('AFFGEOID', axis=1)
# census_df = census_df.drop('GEOID', axis=1)
census_df = census_df.drop('NAME', axis=1)
census_df = census_df.drop('LSAD', axis=1)
census_df = census_df.drop('ALAND', axis=1)
census_df = census_df.drop('AWATER', axis=1)



try:
  os.mkdir(data_dir + "/Census Data/")
except:
  print(data_dir + "/Census Data/" + " Already Exists")


./gdrive/MyDrive/DeliverAI Data Folder/Columbus_mini - RL Delivery Data/Census Data/ Already Exists


In [ ]:
census_df['GEOID'] = census_df['GEOID'].astype(str)
census_df = census_df[census_df['GEOID'].isin(census_tract_subset)] if mini else census_df
census_df.to_file(data_dir + "/Census Data/RAW_census_tract_data.geojson", driver="GeoJSON")

Like MySQL, GeoPandas also provides join operations on data frames using the geometry attribute. Doing a left-join on the locations and census tracts assigns every selected location to a census tract.
Also after joining the 2 data frames, uneccessary attributes from the OSM data are dropped. This significantly reduced file size (by almost 10 times in most cases!)
<br>
<br>
*IMPORTANT NOTE*
<br>
Since we are no longer using the old way of having way and node files for each data location the old named files must no longer be in the folder as the drop files will still have the same name overwriting them but the pick files will have different names than the original processed output causing the original Chicago pickup locations to show up unless the original pickup files are removed.

### Adding Tract ID to Consumer/Producer Data

In [ ]:
# # List all files in the Census Data directory
# census_data_dir = data_dir + "/Census Data/"
# print("Files in Census Data directory:")
# for f in os.listdir(census_data_dir):
#     full_path = os.path.join(census_data_dir, f)
#     if os.path.isfile(full_path):
#         print(f"  {f}: {os.path.getsize(full_path)} bytes")

In [ ]:
census_dir = data_dir + "/Census Data/RAW_census_tract_data.geojson"
census_df = gpd.read_file(census_dir)

# loading directotry of raw pickup locations
pick_data_dir = data_dir + "/Original Location Data/Pickup Point Data/"
pick_files = os.listdir(pick_data_dir)
print(pick_files)
for file_name in pick_files:
  file_dir = pick_data_dir + file_name
  file_df = gpd.read_file(file_dir)


  # edited to make generic crs match
  file_df = file_df.to_crs(census_df.crs)
  #####

  join_left_df = file_df.sjoin(census_df, how="left")
  join_left_df = join_left_df.dropna(subset=['GEOID'])


  for column in join_left_df.columns.tolist():
    if (column not in ["@id", "name", "geometry", "amenity", "building", "shop", "GEOID"]):
      join_left_df = join_left_df.drop(column, axis=1)

  # making file name for storing lat long of all locations
  new_file_dir = data_dir + "/Processed Location Data/Pickup Point Data/" + file_name
  new_file_dir = new_file_dir[:-16]
  new_file_dir += "location_data.geojson"

  join_left_df.to_file(new_file_dir, driver="GeoJSON")

['restaurant_raw_data.geojson', 'supermarket_raw_data.geojson', 'cafe_raw_data.geojson', 'fast_food_raw_data.geojson', 'food_court_raw_data.geojson', 'bakery_raw_data.geojson']


KeyError: ['GEOID']

In [ ]:
census_dir = data_dir + "/Census Data/RAW_census_tract_data.geojson"
census_df = gpd.read_file(census_dir)

# loading directotry of raw drop locations
drop_data_dir = data_dir + "/Original Location Data/Drop Point Data/"
drop_files = os.listdir(drop_data_dir)
print(drop_files)
for file_name in drop_files:
  file_dir = drop_data_dir + file_name
  file_df = gpd.read_file(file_dir)

  # edited to make generic crs match
  file_df = file_df.to_crs(census_df.crs)
  #####


  join_left_df = file_df.sjoin(census_df, how="left")
  join_left_df = join_left_df.dropna(subset=['GEOID'])


  for column in join_left_df.columns.tolist():
    if (column not in ["@id", "name", "geometry", "amenity", "building", "shop", "GEOID"]):
      join_left_df = join_left_df.drop(column, axis=1)

  # making file name for storing lat long of all locations
  new_file_dir = data_dir + "/Processed Location Data/Drop Point Data/" + file_name
  new_file_dir = new_file_dir[:-16]
  new_file_dir += "location_data.geojson"

  join_left_df.to_file(new_file_dir, driver="GeoJSON")

### Find tracts with producers and consumers
These two sections are to get the tracts that have data points within them. This will help to remove all tracts which have no pickup or drop points within them so that they may be discarded to allow for faster data processing.

In [ ]:
#Stores all census tract id's that are used
pick_tracts = set()

pick_data_dir = data_dir + "/Processed Location Data/Pickup Point Data/"
pick_files = os.listdir(pick_data_dir)
print("Files for Pickup Points Data: ",pick_files)

for file_name in pick_files:
  file_dir = pick_data_dir + file_name

  with open(file_dir, 'r') as f:
    data = json.load(f)["features"]

    for element in data:

      properties = element["properties"]
      ID = properties["GEOID"]

      #Used later for tracking which census tracts have points within them to remove tracts without any data points
      pick_tracts.add(ID)

In [ ]:
drop_tracts = set()

drop_data_dir = data_dir + "/Processed Location Data/Drop Point Data/"
drop_files = os.listdir(drop_data_dir)
print("Files for Drop Points Data: ",drop_files)

for file_name in drop_files:
  file_dir = drop_data_dir + file_name

  with open(file_dir, 'r') as f:
    data = json.load(f)["features"]

    for element in data:
      properties = element["properties"]
      ID = properties["GEOID"]

      #Used later for tracking which census tracts have points within them to remove tracts without any data points
      drop_tracts.add(ID)

### Select tracts with producers and consumers and surrounded tracts
This section finally removes all non used census tracts from the census file to reduce its size to just the city that we are focused on.

In [ ]:
# If you want to skip Convex Hull step, run this code block...
census_dir = data_dir + "/Census Data/RAW_census_tract_data.geojson"
census_df = gpd.read_file(census_dir)
census_df.to_file(data_dir + "/Census Data/census_tract_data.geojson", driver="GeoJSON")

In [ ]:
from shapely.geometry import MultiPoint, Polygon, LineString
from scipy.spatial import ConvexHull
import numpy as np

census_dir = data_dir + "/Census Data/RAW_census_tract_data.geojson"
census_df = gpd.read_file(census_dir)

tracts = pick_tracts
for tract in drop_tracts:
  tracts.add(tract)


selected_gdf = census_df[census_df['GEOID'].isin(tracts)]
selected_gdf.to_file(data_dir + "/Census Data/TEST_sel.geojson", driver="GeoJSON")

non_selected_gdf = census_df[~census_df['GEOID'].isin(tracts)]
non_selected_gdf.to_file(data_dir + "/Census Data/TEST_nonsel.geojson", driver="GeoJSON")


# Check and fix invalid geometries
if not selected_gdf.is_valid.all():
    selected_gdf = selected_gdf[selected_gdf.is_valid]
if not non_selected_gdf.is_valid.all():
    non_selected_gdf = non_selected_gdf[non_selected_gdf.is_valid]


# Find the centroid point of each tract
selected_gdf['centroid'] = selected_gdf.geometry.centroid

# Extract the centroid coordinates as a list of tuples
centroid_coords = [(point.x, point.y) for point in selected_gdf['centroid']]

# Calculate the concave hull (alpha shape) with a custom alpha value
alpha_value = 50 # Adjust the alpha value as needed

for alpha in range (1, 20):
    concave_hull = alphashape(centroid_coords, alpha=alpha)
    if (type(concave_hull) != Polygon):
      continue
    alpha_value = alpha



concave_hull = alphashape(centroid_coords, alpha=alpha_value)
surrounded_tracts = gpd.sjoin(census_df, gpd.GeoDataFrame(geometry=[concave_hull]), predicate='intersects')


# Plotting
fig, ax = plt.subplots(figsize=size)

surrounded_tracts.plot(ax=ax, color='red', edgecolor='grey')
selected_gdf.plot(ax=ax, color='lightgrey', edgecolor='grey')
ax.scatter(selected_gdf['centroid'].x, selected_gdf['centroid'].y, color='green', alpha=0.5)
ax.fill(*zip(*concave_hull.exterior.coords), alpha=0.3, edgecolor='blue')
save_image(plt, 'city_removed_tracts')
plt.show()



selected_gdf = pd.concat([selected_gdf, surrounded_tracts])
census_df = census_df[census_df['GEOID'].isin(selected_gdf['GEOID'])]


census_df.to_file(data_dir + "/Census Data/census_tract_data.geojson", driver="GeoJSON")

Add a front end that allows for alpha values to be determined and for tracts to be manually selected and deselected

compile it into an application

In [ ]:
# fig, ax = plt.subplots(figsize=(20, 20))

# L = ['42101008802', '42101008702', '42101008701', '42101008602', '42101008601', '42101008500', '42101008000', '42101007900', '42101009400', '42101009300', '42101009200', '42101010400', '42101010300', '42101010500', '42101010200',
#      '42101008400', '42101007200', '42101008102', '42101008101', '42101007800', '42101007700', '42101007300', '42101011100', '42101011200', '42101019500', '42101011800', '42101011300', '42101010700', '42101010600', '42101011900']
# selected_group = census_df[census_df['GEOID'].isin(L)]

# # census_df.plot(ax=ax, color='lightblue', edgecolor='black')
# selected_group.plot(ax=ax, color='lightgreen', edgecolor='black')

# for idx, row in selected_group.iterrows():
#     centroid = row['geometry'].centroid
#     ax.text(s=row['GEOID'], x=centroid.x, y=centroid.y, horizontalalignment='center', fontsize=5, color='red')

# plt.show()

In [ ]:
# from shapely.ops import nearest_points

# # Step 1: Identify edge geometries in selected_group
# # Create an adjacency matrix
# adjacency_matrix = selected_group.geometry.apply(lambda x: census_df.geometry.touches(x))

# # Identify edge geometries as those that border any geometry outside selected_group
# edge_geometries = selected_group.index[adjacency_matrix.any(axis=1)]

# # Step 2: Find the nearest bordering neighbor for each edge geometry
# nearest_neighbors = []

# for idx in edge_geometries:
#     edge_geometry = selected_group.loc[idx, 'geometry']

#     # Find all geometries in census_df that are not in selected_group
#     remaining_geometries = census_df[~census_df.index.isin(selected_group.index)]

#     # Find the nearest point among bordering neighbors
#     distances = remaining_geometries.geometry.distance(edge_geometry)
#     nearest_neighbor_idx = distances.idxmin()

#     nearest_neighbors.append((idx, nearest_neighbor_idx))

# # Step 3: Plot the map and annotate with nearest neighbors
# fig, ax = plt.subplots(figsize=(20, 20))

# # Plot the selected group on top of the full census data
# selected_group.plot(ax=ax, color='darkblue', edgecolor='black')

# # Annotate the edge geometries with their nearest neighbors
# for edge_idx, neighbor_idx in nearest_neighbors:
#     edge_centroid = selected_group.loc[edge_idx, 'geometry'].centroid
#     neighbor_centroid = census_df.loc[neighbor_idx, 'geometry'].centroid

#     # Draw a line between the edge geometry and its nearest neighbor
#     ax.plot([edge_centroid.x, neighbor_centroid.x],
#             [edge_centroid.y, neighbor_centroid.y], color='red', linestyle='--')

#     # Annotate the edge geometry
#     ax.text(edge_centroid.x, edge_centroid.y, selected_group.loc[edge_idx, 'GEOID'],
#             fontsize=8, color='white', ha='center', va='center')

#     # Annotate the nearest neighbor
#     ax.text(neighbor_centroid.x, neighbor_centroid.y, census_df.loc[neighbor_idx, 'GEOID'],
#             fontsize=8, color='red', ha='center', va='center')

# plt.show()
# print(nearest_neighbors)

# Sorting Consumer/Producer Data

Now that we have the cenus tract each loacation belongs to, it is advisable to store the locations of each census tract seperately. This will help in organising the data by census tract ID. The `amenity`, `building` and `shop` attributes will help in identifying the type of any location.

Locations in the Drop Points are all `ways`. However in the pickup points, they can be a `way` as well as a `node`. The geometry of a `way` is a `Polygon` whereas for `node` it is `Point`. The following code handles both types together. For detailed info about `ways` and `nodes`, kindly read the official wiki docs on [Elements](https://wiki.openstreetmap.org/wiki/Elements).

In [ ]:
census_data = dict()

# loading directotry of sorted by type pickup locations
pick_data_dir = data_dir + "/Processed Location Data/Pickup Point Data/"
pick_files = os.listdir(pick_data_dir)
print("Files for Pickup Points Data: ",pick_files)

for file_name in pick_files:
  file_dir = pick_data_dir + file_name

  with open(file_dir, 'r') as f:
    data = json.load(f)["features"]

    #print(data)

    for element in data:
      properties = element["properties"]
      tract = properties["GEOID"]
      amenity = None
      building = None
      shop = None



      if "amenity" in properties.keys():
        amenity = properties["amenity"]
      if "building" in properties.keys():
        building = properties["building"]
      if "shop" in properties.keys():
        shop = properties["shop"]

      if (element["geometry"]["type"] == "Polygon"):
        coordinates = element["geometry"]["coordinates"][0]
        latitude = np.mean([point[1] for point in coordinates])
        longitude = np.mean([point[0] for point in coordinates])

      elif (element["geometry"]["type"] == "Point"):
        latitude = element["geometry"]["coordinates"][1]
        longitude = element["geometry"]["coordinates"][0]

      if tract not in census_data.keys():
        census_data[tract] = []

      census_data[tract].append({
            "id" : properties["@id"],
            "amenity" : amenity,
            "building" : building,
            "shop": shop,
            "lat" : latitude,
            "lon" : longitude
        })

Repeating the same for drop locations

In [ ]:
# loading directotry of sorted by type drop locations
drop_data_dir = data_dir + "/Processed Location Data/Drop Point Data/"
drop_files = os.listdir(drop_data_dir)
print("Files for Drop Points Data: ",drop_files)

for file_name in drop_files:
  file_dir = drop_data_dir + file_name

  with open(file_dir, 'r') as f:
    data = json.load(f)["features"]

    for element in data:
      properties = element["properties"]
      tract = properties["GEOID"]
      amenity = None
      building = None
      shop = None

      if "amenity" in properties.keys():
        amenity = properties["amenity"]
      if "building" in properties.keys():
        building = properties["building"]
      if "shop" in properties.keys():
        shop = properties["shop"]

      coordinates = element["geometry"]["coordinates"][0]
      latitude = np.mean([point[1] for point in coordinates])
      longitude = np.mean([point[0] for point in coordinates])

      if tract not in census_data.keys():
        census_data[tract] = []

      census_data[tract].append({
            "id" : properties["@id"],
            "amenity" : amenity,
            "building" : building,
            "shop": shop,
            "lat" : latitude,
            "lon" : longitude
        })

This final cell writes the data into a `.json`

In [ ]:
json_object = json.dumps(census_data, indent=4)

with open (data_dir + "/Census Data/census_tract_wise_locations.json", 'w') as f:
  f.write(json_object)

# Finding Hotspot Locations
Based on the analysis, the hotspot location is chosen as center (mean) of the location within the tract. We first store these locations in a seperate `.geojson` file

In [ ]:
from shapely import distance as shapely_distance

with open(data_dir + "/Census Data/census_tract_wise_locations.json", 'r') as f:
  data = json.load(f)

tract_id = list()
hotspot_geometry = list()
num_locations = list()

for census_tract in data:
  lat = []
  lon = []
  for point in data[census_tract]:
    lat.append(point["lat"])
    lon.append(point["lon"])


  lat = pd.Series(lat)
  lon = pd.Series(lon)
  hotspot_lat = lat.mean()
  hotspot_lon = lon.mean()

  tract_id.append(census_tract)
  hotspot_geometry.append(Point(hotspot_lon,hotspot_lat))
  num_locations.append(len(data[census_tract]))



#Adds hotspots to empty tracts
census_df = gpd.read_file(data_dir + "/Census Data/census_tract_data.geojson")


for index, row in census_df.iterrows():
    GEOID = row['GEOID']
    if GEOID not in tract_id:
        centroid = row['geometry'].centroid
        tract_id.append(GEOID)
        hotspot_geometry.append(centroid)
        num_locations.append(0)



try:
  os.mkdir(data_dir + "/Hotspot Data/")
except:
  print(data_dir + "/Hotspot Data/" + " Already Exists")

tract_gdf = gpd.GeoDataFrame({'GEOID': tract_id, 'geometry': hotspot_geometry, 'locations': num_locations}, crs='EPSG:4326')

# IMPORTANT: This line of code below is added to remove an youtlier hotspot that appear far outside city bounds. This issue happened when generating hotspots for LA
#   This is a temporary fix. One of the hotspots generates in the wrong spot, and this removes it.
tract_gdf = tract_gdf[shapely_distance(tract_gdf['geometry'], tract_gdf.unary_union.centroid) <= 5]

tract_gdf.to_file(data_dir + '/Hotspot Data/hotspot_data.geojson', driver='GeoJSON')

Plotting the hotspot locations

In [ ]:
census_dir = data_dir + "/Census Data/census_tract_data.geojson"
census_df = gpd.read_file(census_dir, crs='EPSG:4326')

hotspot_dir = data_dir + "/Hotspot Data/hotspot_data.geojson"
hotspot_locations = gpd.read_file(hotspot_dir, crs='EPSG:4326')

tract = gpd.GeoSeries(census_df.loc[:].geometry)
base = tract.plot(color="white", edgecolor="gray", figsize=size)
hotspot_locations.plot(ax = base, marker='x', color='black', markersize=5, figsize=size)

save_image(plt, 'city_base')
plt.show()

We then make `csv` file to store the frequency of locations within each hotspot.

In [ ]:
with open (data_dir + "/Census Data/census_tract_wise_locations.json", 'r') as f:
  data = json.load(f)

with open (data_dir + "/Census Data/census_frequency.csv", 'w') as f:
  writer = csv.writer(f)
  print(data)
  for census_tract in data:
    writer.writerow([census_tract,len(data[census_tract])])

## Finding Neighbouring Hotspots

In [ ]:
hotspot_dir = data_dir + "/Hotspot Data/hotspot_data.geojson"
hotspot_locations = gpd.read_file(hotspot_dir, crs='EPSG:4326')

projected_hotspots = hotspot_locations.to_crs('EPSG:4326')
distances = projected_hotspots.distance(projected_hotspots.shift())
print(distances[4])

# Superspot Selection

## Loading Hotspot and Census Data

In [ ]:
census_dir = data_dir + "/Census Data/census_tract_data.geojson"
census_df = gpd.read_file(census_dir)

hotspot_dir = data_dir + "/Hotspot Data/hotspot_data.geojson"
hotspot_df = gpd.read_file(hotspot_dir)

pick_data_dir = data_dir + "/Processed Location Data/Pickup Point Data/"
drop_data_dir = data_dir + "/Processed Location Data/Drop Point Data/"


In [ ]:
census_tracts = census_df
hotspots = hotspot_df.drop('locations', axis=1)

# census_tracts = census_df
# hotspots = hotspot_df

## Producer Data

In [ ]:
# First, we get the data for producers. producer_count will store the number of producer locations in each tract
producer_count = defaultdict(int)

pick_files = os.listdir(pick_data_dir)
print("Files for Pickup Points Data: ",pick_files)

for file_name in pick_files:
  file_dir = pick_data_dir + file_name

  with open(file_dir, 'r') as f:
    data = json.load(f)["features"]

    for element in data:
      ID = element["properties"]["GEOID"]

      # if ID in selected_tracts:  # Add this back later
      producer_count[ID] += 1

In [ ]:
# Combine the producer data with the dataframe.
selected_df_prod = census_tracts.copy()
selected_df_prod['producer_count'] = selected_df_prod['GEOID'].apply(lambda x: producer_count.get(str(x), 0))
min_val, max_val = selected_df_prod['producer_count'].min(), selected_df_prod['producer_count'].max()
selected_df_prod['producer_count_normalized'] = (selected_df_prod['producer_count'] - min_val) / (max_val - min_val)

# tract_gdf_prod = tract_gdf.copy()
# tract_gdf_prod['producer_count'] = tract_gdf_prod['GEOID'].apply(lambda x: producer_count.get(str(x), 0))

Plotting Producers

In [ ]:
fig, ax = plt.subplots(figsize=size)

base = selected_df_prod.plot(ax=ax, edgecolor='gray', facecolor="white", linewidth=2, alpha=0.5)

# for idx, row in selected_df_prod.iterrows():
#   gdf = gpd.GeoDataFrame({'geometry': [row['geometry']]})
#   gdf.plot(ax=ax, edgecolor='darkgreen', facecolor="green", linewidth=2, alpha=row['producer_count_normalized'])

#   #points_in_tract = hotspots[hotspots['GEOID'] == row['GEOID']]
#   #points_in_tract.plot(ax=ax, marker='o', color='darkgrey', markersize=80)  # size previously 130

#   label = row['GEOID']
#   x, y = float(points_in_tract.geometry.x.iloc[0]), float(points_in_tract.geometry.y.iloc[0])
#   # ax.annotate(label[-4:], xy=(x, y), xytext=(6, 3), textcoords="offset points", fontsize=8)
#   ax.annotate(str(row['producer_count']), xy=(row['geometry'].centroid.x, row['geometry'].centroid.y), xytext=(3, 3), textcoords="offset points", fontsize=10, color='black')


selected_df_prod.plot(ax=ax, edgecolor='darkgreen', facecolor="green", linewidth=2, alpha=selected_df_prod['producer_count_normalized'])
save_image(plt, 'city_producers')
plt.show()

## Consumer Data

In [ ]:
# First, we get the data for consumers. consumer_count will store the number of consumer locations in each tract
consumer_count = defaultdict(int)

drop_files = os.listdir(drop_data_dir)
print("Files for Drop Points Data: ",drop_files)

for file_name in drop_files:
  file_dir = drop_data_dir + file_name

  with open(file_dir, 'r') as f:
    data = json.load(f)["features"]

    for element in data:
      ID = element["properties"]["GEOID"]

      # if ID in selected_tracts:
      consumer_count[ID] += 1

In [ ]:
# Combine...
selected_df_cons = census_tracts.copy()
selected_df_cons['consumer_count'] = selected_df_cons['GEOID'].apply(lambda x: consumer_count.get(str(x), 0))
min_val, max_val = selected_df_cons['consumer_count'].min(), selected_df_cons['consumer_count'].max()
selected_df_cons['consumer_count_normalized'] = (selected_df_cons['consumer_count'] - min_val) / (max_val - min_val)

Plotting Consumers

In [ ]:
fig, ax = plt.subplots(figsize=size)

base = selected_df_cons.plot(ax=ax, edgecolor='gray', facecolor="white", linewidth=2, alpha=0.5)

'''
# Iterate over GeoDataFrame and get geometry/normalized count for plotting
for idx, row in selected_df_cons.iterrows():
  gdf = gpd.GeoDataFrame({'geometry': [row['geometry']]})
  gdf.plot(ax=ax, edgecolor='darkblue', facecolor="blue", linewidth=2, alpha=row['consumer_count_normalized'])

  #points_in_tract = hotspots[hotspots['GEOID'] == row['GEOID']]
  #points_in_tract.plot(ax=ax, marker='o', color='darkgrey', markersize=130)

  label = row['GEOID']
  x, y = float(points_in_tract.geometry.x.iloc[0]), float(points_in_tract.geometry.y.iloc[0])
  # ax.annotate(label[-4:], xy=(x, y), xytext=(6, 3), textcoords="offset points", fontsize=9)
  ax.annotate(str(row['consumer_count']), xy=(row['geometry'].centroid.x, row['geometry'].centroid.y), xytext=(3, 6), textcoords="offset points", fontsize=12, color='black')
'''

selected_df_cons.plot(ax=ax, edgecolor='darkblue', facecolor="blue", linewidth=2, alpha=selected_df_cons['consumer_count_normalized'])
save_image(plt, 'city_consumers')
plt.show()

## Bordering Data


In [ ]:
selected_df_border = census_tracts.copy()

# Creating fancy spatial join where each tract is compared to every other tract
bordering = gpd.sjoin(selected_df_border, selected_df_border, how='left', predicate='touches')

# Counting number of times each tract ID appears in the spatial join (minus one to exclude the tract itself)
bordering_counts = bordering.groupby('index_right').size()

# Adding this count back to the original GeoDataFrame
selected_df_border['bordering_tracts'] = selected_df_border.index.map(bordering_counts)

# Account for tracts that didn't have any bordering (don't exist in this example)
selected_df_border['bordering_tracts'] = selected_df_border['bordering_tracts'].fillna(0)

min_val, max_val = 0, selected_df_border['bordering_tracts'].max()
selected_df_border['bordering_tracts_normalized'] = (selected_df_border['bordering_tracts'] - min_val) / (max_val - min_val)

Plotting Bordering

In [ ]:

fig, ax = plt.subplots(figsize=size)

base = selected_df_border.plot(ax=ax, edgecolor='gray', facecolor="white", linewidth=2, alpha=0.5)

'''
for idx, row in selected_df_border.iterrows():
  gdf = gpd.GeoDataFrame({'geometry': [row['geometry']]})
  gdf.plot(ax=ax, edgecolor='purple', facecolor="pink", linewidth=2, alpha=row['bordering_tracts_normalized'])

  #points_in_tract = hotspots[hotspots['GEOID'] == row['GEOID']]
  #points_in_tract.plot(ax=ax, marker='o', color='darkgrey', markersize=130)

  label = row['GEOID']
  x, y = float(points_in_tract.geometry.x.iloc[0]), float(points_in_tract.geometry.y.iloc[0])
  ax.annotate(label[-4:], xy=(x, y), xytext=(6, 3), textcoords="offset points", fontsize=9)
  ax.annotate(str(row['bordering_tracts']), xy=(row['geometry'].centroid.x, row['geometry'].centroid.y), xytext=(3, 3), textcoords="offset points", fontsize=12, color='black')
'''

selected_df_border.plot(ax=ax, edgecolor='purple', facecolor="pink", linewidth=2, alpha=selected_df_border['bordering_tracts_normalized'])
save_image(plt, 'city_bordering')
plt.show()

## Scoring Census Tracts
Creating a scoring method for each census tract using these three different parameters (consumers, producers and bordering).

In [ ]:
selected_df_prod['GEOID'] = selected_df_prod['GEOID'].astype(str)
selected_df_cons['GEOID'] = selected_df_cons['GEOID'].astype(str)
selected_df_border['GEOID'] = selected_df_border['GEOID'].astype(str)
selected_df = selected_df_prod.merge(selected_df_cons, on='GEOID').merge(selected_df_border, on='GEOID').drop(['geometry_x', 'geometry_y'], axis=1)  # Join didn't work here for some reason

# Here I am just getting the average of the three normalized values.
selected_df['score'] = ( selected_df['producer_count_normalized'] + selected_df['consumer_count_normalized'] + selected_df['bordering_tracts_normalized'] ) / 3


In [ ]:
# Greate GPD and get best scoring tracts

selected_gdf = gpd.GeoDataFrame(selected_df[['GEOID', 'geometry', 'score']], geometry='geometry')

n = round(len(selected_gdf.index) / superspot_hotspot_ratio, 0)

def get_top_n_no_shared_edges(gdf, n):
  # Sort by score in descending order
  gdf_sorted = gdf.sort_values(by='score', ascending=False).reset_index(drop=True)

  # List to hold the selected geometries and rows
  selected_geometries = []
  skipped_geometries = []
  selected_rows = []

  # Loop through sorted rows
  for index, row in gdf_sorted.iterrows():
    if len(selected_rows) >= n:
      break

    # Check if tract shares an edge with any already selected tract
    share_edge = False
    for selected_geom in selected_geometries:
      if row['geometry'].touches(selected_geom):
        share_edge = True
        skipped_geometries.append(row['GEOID'])
        break

    # If no shared edge, add to the selected list
    if not share_edge:
      selected_geometries.append(row['geometry'])
      selected_rows.append(row)

  # Create a new GeoDataFrame from the selected rows
  return gpd.GeoDataFrame(selected_rows, geometry='geometry'), skipped_geometries

# top_method = input("1. Standard; 2. No shared edges: ")  # NO NEED FOR THIS, WE ARE DOING METHOD 2
top_method = '2'
if top_method == '1':
  top_n_selected_gdf, skipped_geometries = selected_gdf.nlargest(n, 'score'), []
else:
  top_n_selected_gdf, skipped_geometries = get_top_n_no_shared_edges(selected_gdf, n)

In [ ]:
fig, ax = plt.subplots(figsize=size)

base = selected_gdf.plot(ax=ax, edgecolor='gray', facecolor="white", linewidth=2, alpha=0.5)

# Create separate dfs for each type of tract
mask_top_n = selected_gdf['GEOID'].isin(top_n_selected_gdf['GEOID'].values)
mask_skipped = selected_gdf['GEOID'].isin(skipped_geometries)

gdf_top_n = selected_gdf[mask_top_n]
gdf_skipped = selected_gdf[mask_skipped]
gdf_others = selected_gdf[~mask_top_n]


# Make alpha score of superspots slightly higher so they stand out more
alpha_multiplier = 3
gdf_top_n['score'] = gdf_top_n['score'].apply(  lambda x: 1 if x * alpha_multiplier > 1 else x * alpha_multiplier  )

gdf_others.plot(ax=ax, edgecolor='#34a8eb', facecolor="#34d9eb", linewidth=2, alpha=gdf_others['score'])
gdf_top_n.plot(ax=ax, edgecolor='#34a8eb', facecolor="#34eb5b", linewidth=2, alpha=gdf_top_n['score'])
# gdf_skipped.plot(ax=ax, edgecolor='#34a8eb', facecolor="#f5faa0", linewidth=2)

hotspot_df.plot(ax = base, marker='x', color='#2D2D2D', markersize=5, figsize=size)

# for idx, row in selected_gdf.iterrows():
#     centroid = row['geometry'].centroid
#     ax.annotate(
#         row['GEOID'][-4:],
#         xy=(centroid.x, centroid.y),
#         xytext=(-2, 4),
#         textcoords="offset points",
#         fontsize=10,
#         color='black'
#     )

plt.title('Promoted Superspots and Standard Hotspots', fontsize=18)
plt.xlabel("Longitude", fontsize=18)
plt.ylabel("Latitude", fontsize=18)

save_image(plt, 'city_superspots')
plt.show()

#Grouping Tract Areas

In [ ]:
import networkx as nx
from collections import deque

In [ ]:
# @title USING EDGES [DEPRECATED] { display-mode: "form" }
'''
if selected_gdf.index.name != 'GEOID':
    selected_gdf = selected_gdf.set_index('GEOID')
if top_n_selected_gdf.index.name != 'GEOID':
    top_n_selected_gdf = top_n_selected_gdf.set_index('GEOID')

# Add a column to selected_gdf to indicate super census tracts
selected_gdf['is_super'] = selected_gdf.index.isin(top_n_selected_gdf.index)

# Create a graph from the GeoDataFrame
G = nx.Graph()

# Add nodes
for idx, row in selected_gdf.iterrows():
    G.add_node(idx, geometry=row.geometry, is_super=row['is_super'])

# Add edges based on shared boundaries
for idx1, row1 in selected_gdf.iterrows():
    for idx2, row2 in selected_gdf.iterrows():
        if idx1 != idx2 and row1.geometry.touches(row2.geometry):
            G.add_edge(idx1, idx2)

# Initialize the assignment dictionary
assignments = {}

# Assign each super tract to itself
for idx in top_n_selected_gdf.index:
    assignments[idx] = idx

# Initialize queue with non-super tracts (exclude super tracts)
queue = deque(selected_gdf[~selected_gdf['is_super']].index.tolist())

# BFS to assign tracts
while queue:
    current = queue.popleft()
    min_distance = float('inf')
    nearest_super = None
    for neighbor in G.neighbors(current):
        if selected_gdf.loc[neighbor, 'is_super'] and assignments.get(neighbor) is not None:
            # Calculate distance (e.g., use centroid distance or other metric)
            distance = selected_gdf.loc[current, 'geometry'].distance(selected_gdf.loc[neighbor, 'geometry'])
            if distance < min_distance:
                min_distance = distance
                nearest_super = neighbor
    if nearest_super is not None:
        assignments[current] = nearest_super
    else:
        # Handle case where no super tract is found nearby (if needed)
        pass

# Ensure all tracts have been assigned
for idx in selected_gdf.index:
    if idx not in assignments:
        # Assign unassigned tracts to the nearest super tract (fallback strategy)
        nearest_super = None
        min_distance = float('inf')
        for super_idx in top_n_selected_gdf.index:
            distance = selected_gdf.loc[idx, 'geometry'].distance(top_n_selected_gdf.loc[super_idx, 'geometry'])
            if distance < min_distance:
                min_distance = distance
                nearest_super = super_idx
        assignments[idx] = nearest_super

# Add assignments back to the GeoDataFrame
selected_gdf['super_tract'] = selected_gdf.index.map(assignments)
'''

### USING CENTROIDS

In [ ]:
selected_df_prod['GEOID'], selected_df_cons['GEOID'], selected_df_border['GEOID'] = selected_df_prod['GEOID'].astype(str), selected_df_cons['GEOID'].astype(str), selected_df_border['GEOID'].astype(str)
selected_df = selected_df_prod.merge(selected_df_cons, on='GEOID').merge(selected_df_border, on='GEOID').drop(['geometry_x', 'geometry_y'], axis=1)
selected_df['score'] = ( selected_df['producer_count_normalized'] + selected_df['consumer_count_normalized'] + selected_df['bordering_tracts_normalized'] ) / 3
selected_gdf = gpd.GeoDataFrame(selected_df[['GEOID', 'geometry', 'score']], geometry='geometry')


# Ensure GEOID is the index
if selected_gdf.index.name != 'GEOID':
    selected_gdf = selected_gdf.set_index('GEOID')
if top_n_selected_gdf.index.name != 'GEOID':
    top_n_selected_gdf = top_n_selected_gdf.set_index('GEOID')

# Add a column to selected_gdf to indicate super census tracts
selected_gdf['is_super'] = selected_gdf.index.isin(top_n_selected_gdf.index)

# Create a graph from the GeoDataFrame
G = nx.Graph()

# Add nodes with centroid information
for idx, row in selected_gdf.iterrows():
    G.add_node(idx, geometry=row.geometry, centroid=row.geometry.centroid, is_super=row['is_super'])

# Add edges based on shared boundaries
for idx1, row1 in selected_gdf.iterrows():
    for idx2, row2 in selected_gdf.iterrows():
        if idx1 != idx2 and row1.geometry.touches(row2.geometry):
            G.add_edge(idx1, idx2)

# Initialize the assignment dictionary and children tracking
assignments = {}
children = {idx: [] for idx in top_n_selected_gdf.index}

# Assign each super tract to itself
for idx in top_n_selected_gdf.index:
    assignments[idx] = idx

# Initialize queue with non-super tracts (exclude super tracts)
queue = deque(selected_gdf[~selected_gdf['is_super']].index.tolist())

# BFS to assign tracts based on centroid distance
while queue:
    current = queue.popleft()
    min_distance = float('inf')
    nearest_super = None
    current_centroid = selected_gdf.loc[current, 'geometry'].centroid
    for neighbor in G.neighbors(current):
        if selected_gdf.loc[neighbor, 'is_super'] and assignments.get(neighbor) is not None:
            neighbor_centroid = selected_gdf.loc[neighbor, 'geometry'].centroid
            distance = current_centroid.distance(neighbor_centroid)
            if distance < min_distance:
                min_distance = distance
                nearest_super = neighbor
    if nearest_super is not None:
        assignments[current] = nearest_super
        children[nearest_super].append(current)
    else:
        # Handle case where no super tract is found nearby (if needed)
        pass

# Ensure all tracts have been assigned
for idx in selected_gdf.index:
    if idx not in assignments:
        nearest_super = None
        min_distance = float('inf')
        current_centroid = selected_gdf.loc[idx, 'geometry'].centroid
        for super_idx in top_n_selected_gdf.index:
            super_centroid = top_n_selected_gdf.loc[super_idx, 'geometry'].centroid
            distance = current_centroid.distance(super_centroid)
            if distance < min_distance:
                min_distance = distance
                nearest_super = super_idx
        assignments[idx] = nearest_super
        children[nearest_super].append(idx)

# Remove super tracts with fewer than 3 children
super_tracts_to_remove = [idx for idx, child_list in children.items() if len(child_list) < min_children]

for super_tract in super_tracts_to_remove:
    # Collect tracts to be reassigned
    tracts_to_reassign = [super_tract] + children[super_tract]
    for tract in tracts_to_reassign:
        min_distance = float('inf')
        nearest_super = None
        current_centroid = selected_gdf.loc[tract, 'geometry'].centroid
        for remaining_super in set(top_n_selected_gdf.index) - set(super_tracts_to_remove):
            super_centroid = selected_gdf.loc[remaining_super, 'geometry'].centroid
            distance = current_centroid.distance(super_centroid)
            if distance < min_distance:
                min_distance = distance
                nearest_super = remaining_super
        assignments[tract] = nearest_super
        children[nearest_super].append(tract)

    # Mark the super tract as non-super
    selected_gdf.at[super_tract, 'is_super'] = False

# Remove reassigned tracts from their original super tracts' children lists
for super_tract in super_tracts_to_remove:
    del children[super_tract]

# Add assignments and children back to the GeoDataFrame
selected_gdf['super_tract'] = selected_gdf.index.map(assignments)

# Initialize the 'children' column for super tracts
selected_gdf['children'] = selected_gdf.index.map(lambda idx: children.get(idx, []))

# Ensure non-super tracts have an empty list as children
#selected_gdf.loc[~selected_gdf['is_super'], 'children'] = [[] for _ in range(len(selected_gdf[~selected_gdf['is_super']]))]


# Convert children lists to JSON strings
selected_gdf['children'] = selected_gdf['children'].apply(json.dumps)

# Save to GeoJSON
try:
  os.mkdir(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}")
except:
  print(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}" + "Already exists")

selected_gdf.to_file(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/layered_hotspots.geojson", driver='GeoJSON')

###Plotting

In [ ]:
from matplotlib.colors import ListedColormap
from statistics import median

min_children_dir = data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/layered_hotspots.geojson"
selected_gdf = gpd.read_file(min_children_dir).to_crs(epsg=3395)


selected_gdf = selected_gdf.to_crs(epsg=3395)


#Plotting
fig, ax = plt.subplots(figsize=size)

# Get unique super tracts
unique_super_tracts = selected_gdf['super_tract'].unique()

# Generate a color map with enough unique colors
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_super_tracts)))
color_map = ListedColormap(colors)

# Plot each super tract and its corresponding non-super tracts
for i, super_tract in enumerate(unique_super_tracts):
    # Plot non-super tracts controlled by the current super tract
    non_super_gdf = selected_gdf[(selected_gdf['super_tract'] == super_tract) & (~selected_gdf['is_super'])]
    if not non_super_gdf.empty:
        non_super_gdf.plot(color=color_map(i), alpha=0.2, edgecolor='black', ax=ax, label=f'Controlled by {super_tract}')

    # Plot the current super tract
    super_gdf = selected_gdf[(selected_gdf['super_tract'] == super_tract) & (selected_gdf['is_super'])]
    if not super_gdf.empty:
        super_gdf.plot(color=color_map(i), alpha=1.0, edgecolor='black', ax=ax)

        # Plot black dots at the centroid of super tracts
        for idx, row in super_gdf.iterrows():
            if row.geometry.is_empty or row.geometry.centroid.is_empty:
                continue
            centroid = row.geometry.centroid
            ax.plot(centroid.x, centroid.y, 'ko')

            # Plot lines from super tract centroid to controlled tracts centroids
            controlled_centroids = non_super_gdf.geometry.centroid
            for control_centroid in controlled_centroids:
                ax.plot([centroid.x, control_centroid.x], [centroid.y, control_centroid.y], 'k-', alpha=0.5)

# Set aspect ratio
ax.set_aspect('equal', 'box')
plt.title('Assigned Superspots and Controlled Hotspots', fontsize=18)
plt.xlabel("Longitude", fontsize=18)
plt.ylabel("Latitude", fontsize=18)

save_image(plt, f'city_grouped_{min_children}_{superspot_hotspot_ratio}')
plt.show()

In [ ]:
# Step 1: Read the CSV file containing GEOID and the number of locations
census_frequency = pd.read_csv(data_dir + '/Census Data/census_frequency.csv', header=None, names=['GEOID', 'locations'], dtype={'GEOID': str})

# Ensure GEOID is the index in the census frequency data
census_frequency['GEOID'] = census_frequency['GEOID'].astype(str).str.strip()
census_frequency = census_frequency.set_index('GEOID')
# Step 2: Read the GeoJSON file and convert 'children' field from JSON strings back to lists
gdf = gpd.read_file(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/layered_hotspots.geojson")


# Ensure GEOID consistency
gdf['GEOID'] = gdf['GEOID'].astype(str).str.strip()
gdf = gdf.set_index('GEOID')

gdf['children'] = gdf['children'].apply(json.loads)

# Step 3: Merge the location data into the GeoDataFrame
gdf['locations'] = gdf.index.map(census_frequency['locations'])



# Fill NaN values with 0 (for tracts not in the census_frequency file)
gdf['locations'] = gdf['locations'].fillna(0)

# Step 4: Calculate the total number of locations for each super tract including its children
def calculate_total_locations(row):
    if row['is_super']:
        total_locations = row['locations']
        for child in row['children']:
            child = str(child).strip()  # Ensure the child GEOID is formatted correctly
            if child in gdf.index:
                total_locations += gdf.at[child, 'locations']
            else:
                print(f"Warning: GEOID {child} not found in the DataFrame.")
        return total_locations
    return 0

gdf['total_locations'] = gdf.apply(calculate_total_locations, axis=1)


# Step 5: Create gdf_plot with each child inheriting the total locations of its parent super tract
gdf_plot = gdf.copy()

for idx, row in gdf_plot.iterrows():
    if not row['is_super']:
        super_tract_total_locations = gdf_plot.at[row['super_tract'], 'total_locations']
        gdf_plot.at[idx, 'total_locations'] = super_tract_total_locations

# Step 5: Plotting the heat map
fig, ax = plt.subplots(figsize=size)

# Plot boundaries for context
gdf_plot.boundary.plot(ax=ax, linewidth=1, color='black')

gdf_plot_plot = gdf_plot.plot(
    column='total_locations',
    cmap='OrRd',
    linewidth=0.8,
    ax=ax,
    edgecolor='0.8',
    legend=True,
    legend_kwds={'shrink': 0.8}  # <-- shrink makes the colorbar shorter
)
# Plot heat map based on total locations
#gdf_plot.plot(column='total_locations', cmap='OrRd', linewidth=0.8, ax=ax, edgecolor='0.8', legend=True)

# Plot super tracts boundaries in bold
super_tracts = gdf[gdf['is_super']]


# Adding labels to super tracts with total locations
for idx, row in super_tracts.iterrows():
    plt.text(row.geometry.centroid.x, row.geometry.centroid.y, f"{int(row['total_locations'])}",
             fontsize=10, ha='center', color='black', weight='bold')

plt.title("Heat Map of Total Locations in Each Super Tract", fontsize=18)
plt.xlabel("Longitude", fontsize=18)
plt.ylabel("Latitude", fontsize=18)

save_image(plt, f'city_heatmap_{min_children}_{superspot_hotspot_ratio}')
plt.show()

gdf['children'] = gdf['children'].apply(json.dumps)
gdf.to_file(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/layered_hotspots.geojson", driver='GeoJSON')

In [ ]:
print(f"""
MAP STATS:
  Total  Locations\t\t\t= {gdf['locations'].sum()}
  Mean   Locations\t[per hotspot]\t= {round(gdf['locations'].mean(), 1)}
  Median Locations\t[per hotspot]\t= {int(gdf['locations'].median())}
    {'-'*50}
  Total  Producers\t\t\t= {selected_df_prod['producer_count'].sum()}
  Mean   Producers\t[per hotspot]\t= {round(selected_df_prod['producer_count'].mean(), 1)}
  Median Producers\t[per hotspot]\t= {int(selected_df_prod['producer_count'].median())}
    {'-'*50}
  Total  Consumers\t\t\t= {selected_df_cons['consumer_count'].sum()}
  Mean   Consumers\t[per hotspot]\t= {round(selected_df_cons['consumer_count'].mean(), 1)}
  Median Consumers\t[per hotspot]\t= {int(selected_df_cons['consumer_count'].median())}
    {'-'*50}
  Mean   Bordering\t[per hotspot]\t= {round(selected_df_border['bordering_tracts'].mean(), 1)}
  Median Bordering\t[per hotspot]\t= {int(selected_df_border['bordering_tracts'].median())}
    {'-'*50}
  Num.   Hotspots\t\t\t= {len(gdf.index)}
  Num.   Super-spots\t\t\t= {len(gdf[gdf['is_super']].index)}
    {'-'*50}
  Median Child-Hotspots\t\t\t= {int(median(list(map(lambda l: len(l), [ast.literal_eval(ch) for ch in gdf[gdf['is_super']]['children']]))))}
  Mean   Locations\t[per cluster]\t= {round(gdf[gdf['is_super']]['total_locations'].mean(), 2)}
  Median Locations\t[per cluster]\t= {int(gdf[gdf['is_super']]['total_locations'].median())}
""")


# Save this information to a txt file
map_summary_data = f"""
Total  Locations\t:\t{gdf['locations'].sum()}
Mean   Locations\t[per hotspot]\t:\t{round(gdf['locations'].mean(), 1)}
Median Locations\t[per hotspot]\t:\t{int(gdf['locations'].median())}
Total  Producers\t:\t{selected_df_prod['producer_count'].sum()}
Mean   Producers\t[per hotspot]\t:\t{round(selected_df_prod['producer_count'].mean(), 1)}
Median Producers\t[per hotspot]\t:\t{int(selected_df_prod['producer_count'].median())}
Total  Consumers\t:\t{selected_df_cons['consumer_count'].sum()}
Mean   Consumers\t[per hotspot]\t:\t{round(selected_df_cons['consumer_count'].mean(), 1)}
Median Consumers\t[per hotspot]\t:\t{int(selected_df_cons['consumer_count'].median())}
Mean   Bordering\t[per hotspot]\t:\t{round(selected_df_border['bordering_tracts'].mean(), 1)}
Median Bordering\t[per hotspot]\t:\t{int(selected_df_border['bordering_tracts'].median())}
Num.   Hotspots\t:\t{len(gdf.index)}
Num.   Super-spots\t:\t{len(gdf[gdf['is_super']].index)}
Median Child-Hotspots\t:\t{int(median(list(map(lambda l: len(l), [ast.literal_eval(ch) for ch in gdf[gdf['is_super']]['children']]))))}
Mean   Locations\t[per cluster]\t:\t{round(gdf[gdf['is_super']]['total_locations'].mean(), 2)}
Median Locations\t[per cluster]\t:\t{int(gdf[gdf['is_super']]['total_locations'].median())}
"""

with open(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/map_summary_data.txt", 'w') as f:
  f.write(map_summary_data)

TODO
* setup checks for files so this can all be run at once
* setup option for graph generation



# Getting Adjacency Matrix

## Setup

In [ ]:
# Create data to be used. Make sure you run the Grouping Tract Areas code before this so that the selected_gdf dataframe is created properly.
gh_df = pd.DataFrame(selected_gdf.drop(['geometry', 'score'], axis=1))
gh_df = gh_df.merge(hotspots[['GEOID', 'geometry']], on='GEOID', how='left')

gh_df['x'] = gh_df['geometry'].apply(lambda geom: geom.x).astype(float)
gh_df['y'] = gh_df['geometry'].apply(lambda geom: geom.y).astype(float)

gh_df = gh_df.drop('geometry', axis=1)
gh_df['children'] = gh_df['children'].apply(ast.literal_eval)
gh_df.to_csv(data_dir + f'/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/graphhopper_dataframe.csv')

# gh_df should have columns:
# 	GEOID	is_super	super_tract	children	x	y

In [ ]:
# OR JUST IMPORT gh_df here
gh_df = pd.read_csv(data_dir + f'/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/graphhopper_dataframe.csv', dtype={'GEOID': 'string', 'super_tract': 'string'}).drop(["Unnamed: 0"], axis=1)
gh_df['children'] = gh_df['children'].apply(ast.literal_eval)

In [ ]:
# The keys used in the query

keys = [
  'd927fb04-31ab-43e6-9b4b-62a11d96e928',
  '1b08644e-8836-4d6b-86b6-22ed8b02a4c6'
]

# Additional keys in reserve:
#

key_ind = len(keys)-1  # Index in the list that determines which key is being used. Starts from last and works its way up because I feel cool when I do it that way



URL = "https://graphhopper.com/api/1/matrix"  # GraphHopper matrix API url
req_lim = 80  # Limit on number of requests in one go



# Generates GraphHopper query using provided params
def gh_query(from_point, to_point, keys: list):
  global key_ind

  # Do-while loop in python, runs query before checking
  while True:
    # Query to provide the key
    query = { "key": keys[key_ind] }

    # Payload to send rest of data
    payload = {
        "profile": "car",
        "from_points": from_point,
        "to_points": to_point,
        "out_arrays": ["distances","times","weights"],
        "fail_fast": "true",
      }

    # Get response using POST
    response = requests.post(URL, json=payload, params=query)
    data = response.json()  # Store data as json

    # If there is a message, then something went wrong.
    if 'message' not in data.keys():
      return data  # No message so return data
    else:  # Assume message says key ran out
      if key_ind > 0:  # Make sure we don't exit bounds of list
        key_ind -= 1
      else:  # Otherwise we have run out of keys
        if 'Minutely API limit heavily violated' not in data['message']:
          print(data['message'])
        # raise Exception("Error Message returned")
        key_ind = len(keys)-1

  return None  # Function shouldn't reach this but it's here just in case since the only other return is in an if-statement

gh_df_super = gh_df[gh_df['is_super']]  # Dataframe of only super-spots
geoid_ind = {id_: index for index, id_ in enumerate(gh_df['GEOID'].unique())}  # Dictionary with GEOIDs as keys and numbers 0 - len(gh_df['GEOID'].unique()) as values

In [ ]:
ind_geoid = {v: k for k, v in geoid_ind.items()}

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 500)
pd.set_option('display.colheader_justify', 'center')
pd.set_option('display.precision', 2)

def print_matrix(matrix):
    global ind_geoid

    modified_matrix = np.where(matrix > 1000000, -1, matrix)

    index_labels = [ind_geoid[i][-4:] for i in range(len(matrix))]  # Setting row labels
    columns_labels = [ind_geoid[i][-4:] for i in range(len(matrix))]  # Setting col labels

    print(pd.DataFrame(modified_matrix, columns=columns_labels, index=index_labels))

## Hotspot to Hotspot

IMPORTANT! Some of this code is structured in a way such that it avoids re-creating data from scratch. If you want to regenerate the matrices from scratch make some adjustments to the code block below.

In [ ]:
with open(data_dir + "/Hotspot Data/map_geoid_index.json", 'w') as f:
    json.dump(geoid_ind, f)

# Create a default matrix of -1 values for distances, times, weights
if 'distance_adjacency_matrix.npy' not in os.listdir(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/"):
  print("created new")
  distances = np.full((len(gh_df.index), len(gh_df.index)), sys.maxsize)
  times = np.full((len(gh_df.index), len(gh_df.index)), sys.maxsize)
  weights = np.full((len(gh_df.index), len(gh_df.index)), sys.maxsize)
else:
  print("loaded existing")
  distances = np.load(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/distance_adjacency_matrix.npy")
  times = np.load(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/time_adjacency_matrix.npy")
  weights = np.load(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/weight_adjacency_matrix.npy")

Matrix with removed edges.

In [ ]:
modified_mask = (sys.maxsize - np.mean(distances, axis=1)) > 0

# Set the last occurring True to False, if any True exists
#   This is to prevent issues with partially complete rows
true_indices = np.where(modified_mask)[0]
if true_indices.size > 0:
  modified_mask[true_indices[-1]] = False

print(len(true_indices), "already complete,", (len(modified_mask) - len(true_indices)), "left to go.")

for _, row in tqdm(gh_df[~modified_mask].iterrows(), desc="Tracts"):

  is_super = row['is_super']  # Bool statement that tells if row is superspot
  from_points = [ [ row['x'], row['y'] ] ]  # From point, is just the row. All distances will be measured from here.

  if is_super:  # If it is a super-spot...

    # Super-spot iteration, block by block with two pointers, size of block depends on req_limit by graphhopper
    super_start = 0

    # While not exceeding length of super-spot dataframe
    while super_start < len(gh_df_super.index):
      super_end = min(super_start + req_lim, len(gh_df_super.index))

      to_points = gh_df_super.iloc[super_start:super_end][['x', 'y']].values.tolist()  # All superspot points to travel to. Obtained block by block using pointers
      to_points_geoids = gh_df_super.iloc[super_start:super_end]['GEOID'].values.tolist()  # IDs of all superspot points to travel to.

      data = gh_query(from_points, to_points, keys)  # Get data between it and all super-spots. Function is defined in previous block

      # Save data below. Note that the order in which the to_points were provided is the same as the output (I tested this), so we can just zip like this and the keys/values will match.
      for key, distance, time, weight in zip(to_points_geoids, data['distances'][0], data['times'][0], data['weights'][0]):  # Zips all values in for loop
        distances[geoid_ind[row['GEOID']], geoid_ind[key]] = distance  # Stores in appropriate index in distances matrix using dicts to get the right indexes
        times[geoid_ind[row['GEOID']], geoid_ind[key]] = time  # Same for time
        weights[geoid_ind[row['GEOID']], geoid_ind[key]] = weight  # Etc...

      # Update pointers
      super_start = super_end

  # Get data between it and neighbors (a super-spot is considered a "neighbor" of its children in this implementation)
  neighbors_list = list(gh_df[gh_df['GEOID'] == row['super_tract']]['children'])[0].copy()
  neighbors_list.append(str(row['super_tract']))
  neighbors_df = gh_df[gh_df['GEOID'].isin(neighbors_list)]

  len_neighbors = len(neighbors_list)  # Length of neighbors list, is used frequently in the code below.

  # Pointers for neighbor list (most likely not necessary as there shouldn't be anywhere close to 80 neighbors in a group)
  neighbor_start = 0

  # While not exceeding length of neighbor hotspots
  while neighbor_start < len_neighbors:
    neighbor_end = min(neighbor_start + req_lim, len_neighbors)

    to_points = neighbors_df.iloc[neighbor_start:neighbor_end][['x', 'y']].values.tolist()  # All hotspots to travel to.
    to_points_geoids = neighbors_df.iloc[neighbor_start:neighbor_end]['GEOID'].values.tolist()  # IDs of all hotspots to travel to

    data = gh_query(from_points, to_points, keys)  # Get data

    # Save data below. Same method as in super-spots (seen above)
    for key, distance, time, weight in zip(to_points_geoids, data['distances'][0], data['times'][0], data['weights'][0]):
      distances[geoid_ind[row['GEOID']], geoid_ind[key]] = distance
      times[geoid_ind[row['GEOID']], geoid_ind[key]] = time
      weights[geoid_ind[row['GEOID']], geoid_ind[key]] = weight

    # Update pointers
    neighbor_start = neighbor_end

  np.save(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/distance_adjacency_matrix.npy", distances)
  np.save(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/time_adjacency_matrix.npy", times)
  np.save(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/weight_adjacency_matrix.npy", weights)


np.save(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/distance_adjacency_matrix.npy", distances)
np.save(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/time_adjacency_matrix.npy", times)
np.save(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/weight_adjacency_matrix.npy", weights)

In [ ]:
print_matrix(distances)

In [ ]:
neighbors = []
for _, row in gh_df.iterrows():
  neighbors = list(gh_df[gh_df['GEOID'] == row['super_tract']]['children'])[0].copy()
  neighbors.append(row['super_tract'])

  if row['is_super']:
    neighbors += list(gh_df_super['GEOID']).copy()

  for n in neighbors:
    if distances[geoid_ind[row['GEOID']], geoid_ind[n]] == sys.maxsize:
      print("No edge between", row['GEOID'], n)

## Hotspot to Locations within Tract

In [ ]:
hotspots = gpd.read_file(data_dir + "/Hotspot Data/hotspot_data.geojson").drop('locations', axis=1)

In [ ]:
with open (data_dir + "/Census Data/census_tract_wise_locations.json", 'r') as f:
  data = json.load(f)

hotspot_points = dict()
for hotspot in gh_df['GEOID'].unique():
  hotspot_points[hotspot] = list()

for hotspot in gh_df['GEOID'].unique():
  if hotspot in data.keys():
    for loc in data[hotspot]:
      hotspot_points[hotspot].append([float(loc['lon']), float(loc['lat'])])

In [ ]:
def check_list_equality(li1, li2):
  dif1 = np.setdiff1d(li1, li2)
  dif2 = np.setdiff1d(li2, li1)

  temp3 = np.concatenate((dif1, dif2))
  print(dif1)
  print(dif2)
  print(list(temp3) == [])  # THIS SHOULD BE TRUE!!

check_list_equality(
  list(hotspot_points.keys()),
  list(gh_df['GEOID'].unique())
)

In [ ]:
if os.path.exists(data_dir + "/avg_hotspot_data.json"):
  print("Loaded existing")
  with open(data_dir + "/avg_hotspot_data.json", 'r') as f:
    avg_hotspot_data = json.load(f)
else:
  print("Made new")
  avg_hotspot_data = dict()

In [ ]:
import time
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

url = "https://graphhopper.com/api/1/matrix"

census_count = len(avg_hotspot_data.keys())
for hotspot in gh_df['GEOID'].unique():  # gh_df, or gh_df[gh_df.index > 545]
  if hotspot in avg_hotspot_data.keys():
    continue
  # if avg_hotspot_data[hotspot]['distance'] <= 2000:
  #   continue

  points = hotspot_points[hotspot]
  print(f"!!! {census_count}. Num points: {len(points)}")

  hotspot_point = [hotspots[hotspots['GEOID'] == hotspot]][0]['geometry']
  from_point = [[float(hotspot_point.x), float(hotspot_point.y)]]

  # Always create the dictionary
  avg_hotspot_data[hotspot] = dict()
  avg_hotspot_data[hotspot]["distance"] = 0
  avg_hotspot_data[hotspot]["time"] = 0


  max_value = 160  # len(points)
  start_p = 0

  dynamic_max_value = min(int(len(points)), int(max_value))
  while start_p < dynamic_max_value:
    end_p = min(start_p + req_lim, dynamic_max_value)
    print(start_p, end_p, end="\t||\t")

    data = gh_query(from_point, hotspot_points[hotspot][start_p:end_p], keys)


    # if start_p == 0:
    #   avg_hotspot_data[hotspot] = dict()
    #   avg_hotspot_data[hotspot]["distance"] = sum(data['distances'][0])/len(data['distances'][0])
    #   avg_hotspot_data[hotspot]["time"] = sum(data['times'][0])/len(data['times'][0])
    # else:
    avg_hotspot_data[hotspot]["distance"] += sum(data['distances'][0])
    avg_hotspot_data[hotspot]["time"] += sum(data['times'][0])

    start_p = end_p

  census_count += 1

  avg_hotspot_data[hotspot]["distance"] /= max(1, dynamic_max_value)
  avg_hotspot_data[hotspot]["time"] /= max(1, dynamic_max_value)

  json_object = json.dumps(avg_hotspot_data, indent=4)
  with open (data_dir + "/avg_hotspot_data.json", 'w') as f:
    f.write(json_object)
  print()


json_object = json.dumps(avg_hotspot_data, indent=4)
with open (data_dir + "/avg_hotspot_data.json", 'w') as f:
  f.write(json_object)

In [ ]:
check_list_equality(
  list(avg_hotspot_data.keys()),
  list(gh_df['GEOID'].unique())
)

In [ ]:
min_children_dir = data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/layered_hotspots.geojson"
selected_gdf = gpd.read_file(min_children_dir).to_crs(epsg=3395)

large_tracts = [k for k, v in avg_hotspot_data.items() if v['distance'] > 2000]

In [ ]:
temp_gdf = selected_gdf[selected_gdf['GEOID'].isin(large_tracts)]

fig, ax = plt.subplots(figsize=(10, 10))
selected_gdf.plot(ax=ax, color='grey')
temp_gdf.plot(ax=ax, color='red', edgecolor='white')

In [ ]:
{k: v for k, v in avg_hotspot_data.items() if v['distance'] > 2000}

In [ ]:
temp_sum_dist = 0
temp_sum_time = 0

hotspot = '39049006353'
points = hotspot_points[hotspot]

hotspot_point = [hotspots[hotspots['GEOID'] == hotspot]][0]['geometry']
from_point = [[float(hotspot_point.x), float(hotspot_point.y)]]

max_value = 80  # len(points) or 80
start_p = 0

dynamic_max_value = min(int(len(points)), int(max_value))
while start_p < dynamic_max_value:
  end_p = min(start_p + req_lim, dynamic_max_value)
  print(start_p, end_p, end="\t||\t")

  data = gh_query(from_point, hotspot_points[hotspot][start_p:end_p], keys)


  # if start_p == 0:
  #   avg_hotspot_data[hotspot] = dict()
  #   avg_hotspot_data[hotspot]["distance"] = sum(data['distances'][0])/len(data['distances'][0])
  #   avg_hotspot_data[hotspot]["time"] = sum(data['times'][0])/len(data['times'][0])
  # else:
  temp_sum_dist += sum(data['distances'][0])
  temp_sum_time += sum(data['times'][0])

  start_p = end_p

print(temp_sum_dist)
print(temp_sum_time)
temp_sum_dist /= max(1, dynamic_max_value)
temp_sum_time /= max(1, dynamic_max_value)

print(temp_sum_dist)
print(temp_sum_time)

In [ ]:
gh_df[gh_df['GEOID'] == '39049006353']